In [ ]:
pip install earthengine-api geemap pycrs

## Preparations

Import libraries

In [ ]:
import ee
import geemap
from pathlib import Path
import geopandas as gpd

Authenticate Google Earth Engine

In [ ]:
ee.Authenticate() 
ee.Initialize(project='your-ee-project-name', opt_url='https://earthengine-highvolume.googleapis.com')

geemap requires .shp or .geojson extension input \
\
`field_id` is need to identify predictions with real fields

In [9]:
input_file = "../data/raw/fields.fgb"
output_file = "../data/raw/fields.shp"

gdf = gpd.read_file(input_file)

gdf = gdf.reset_index(drop=True)
gdf["field_id"] = [
    i+1 for i in range(len(gdf))
]
gdf = gdf.to_crs('epsg:4326')

gdf.to_file(output_file)

print(f"saved {output_file}")

saved ../data/raw/fields.shp


## Downloading data from Google Earth Engine

Download spectral data (Landat 5, 8, 9)

In [ ]:
def mask_l8_clouds(image):
    """Masks clouds and cloud shadows in a Landsat 8 SR image using QA_PIXEL.

  Args:
    image (ee.Image): A Landsat 8 SR image (Collection 2, Level 2).

  Returns:
    ee.Image: A cloud-masked Landsat 8 image with scaled reflectance
              and original metadata (including system:time_start).
  """
    qa = image.select('QA_PIXEL')
    
    dilated_cloud_bit_mask = 1 << 1
    cloud_shadow_bit_mask = 1 << 4
    cloud_bit_mask = 1 << 3
    snow_mask= 1 << 5
    mask = (
      qa.bitwiseAnd(dilated_cloud_bit_mask).eq(0)
      .And(qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0))
      .And(qa.bitwiseAnd(cloud_bit_mask).eq(0))
    )
    return image.updateMask(mask)\
      .select("SR_B.*").divide(10000)\
        .set('date', image.date().format('YYYY-MM-dd')).copyProperties(image, ["system:time_start", "system:index"]) 

    
def add_indices_landsat(image):
    red=image.select('SR_B4').rename('red')
    nir=image.select('SR_B5').rename('nir')
    blue=image.select('SR_B2').rename('blue')
    swir1=image.select('SR_B6').rename('swir1')
    green=image.select('SR_B3').rename('green')
    swir2=image.select('SR_B7').rename('swir2')
 
    return image.addBands([
        red, nir, blue, swir1, green, swir2
    ])

def add_indices_landsat5(image):
    red = image.select('SR_B3').rename('red')
    nir = image.select('SR_B4').rename('nir')
    blue = image.select('SR_B1').rename('blue')
    green = image.select('SR_B2').rename('green')
    swir1 = image.select('SR_B5').rename('swir1')
    swir2 = image.select('SR_B7').rename('swir2')
    return image.addBands([red, nir, blue, swir1, green, swir2])


file='../data/raw/fields.shp'
output_file='../data/raw/fields_spectral_data.csv'

year=2015

shape = geemap.shp_to_ee(file)

start_date = f"{year}-04-01"
end_date = f"{year}-11-01"

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterDate(start_date, end_date)
      .filterBounds(shape)
      .filter(ee.Filter.lt('CLOUD_COVER', 50))
      .map(mask_l8_clouds)
      .map(add_indices_landsat))

l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
      .filterDate(start_date, end_date)
      .filterBounds(shape)
      .filter(ee.Filter.lt('CLOUD_COVER', 50))
      .map(mask_l8_clouds)
      .map(add_indices_landsat5))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
      .filterDate(start_date, end_date)
      .filterBounds(shape)
      .filter(ee.Filter.lt('CLOUD_COVER', 50))
      .map(mask_l8_clouds)
      .map(add_indices_landsat))

collection = l5.merge(l8).merge(l9)

count = collection.size().getInfo()

print(f"Processing {file}: found {count} images.")


try:
    band_collection = collection.select(['red', 'nir', 'blue', 'swir1', 'green', 'swir2'])
    time_series = band_collection.toBands()
    

    geemap.zonal_statistics(
        time_series, 
        shape, 
        output_file, 
        statistics_type='median', 
        scale=30
    )

except Exception as e:
    print(f"Unexpected error for {file}: {e}")

print(f"All the data downloaded")

Download meteorological data (ERA5)

In [14]:
def filter_bands(image):
    """Filter images adding temperature and precipitation"""
    try:
        temperature = image.select('temperature_2m').rename('temperature')
    except Exception as e:
        temperature = image.constant(None).rename('temperature')
        logging.warning(f"No temperature band found in image: {image.id().getInfo()}. Error: {e}")

    try:
        precipitation = image.select('total_precipitation_sum').rename('precipitation')
    except Exception as e:
        precipitation = image.constant(None).rename('precipitation')
        logging.warning(f"No precipitation band found in image: {image.id().getInfo()}. Error: {e}")

    return image.addBands([temperature, precipitation]).set('date', image.date().format('YYYY-MM-dd'))


file='../data/raw/fields.shp'
output_file='../data/raw/fields_meteo_data.csv'

year=2015

shape = geemap.shp_to_ee(file)
start_date = f"{year}-04-01"
end_date = f"{year}-11-01"

collection=(ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
            .filter(ee.Filter.date(start_date, end_date))
            .filterBounds(shape)
                    .map(filter_bands))


count = collection.size().getInfo()

print(f"Processing {file}: found {count} images.")

try:
    meteodata = (collection.select(['temperature', 'precipitation'])).toBands()

    geemap.zonal_statistics(
        meteodata, 
        shape, 
        output_file, 
        statistics_type='median', 
        scale=100
    )
except Exception as e:
    print(f"Unexpected error for {file}: {e}")

print(f"All the data downloaded")

Processing ../data/raw/fields.shp: found 214 images.
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/work/CropGRM/data/raw/fields_meteo_data.csv
All the data downloaded
